# 03 — Cálculo da variável-alvo Y: `status_real`

**Projeto:** Preditor de Falhas ML (Grupo 16)  
**Disciplinas:** ED2, Redes e APS  
**Turma:** Ciência da Computação — 3.º e 4.º semestres  
**Entrega:** cálculo e documentação do rótulo `status_real`

> Este notebook documenta e demonstra o cálculo solicitado para a disciplina. A regra oficial continua na função `status_real` do pacote, em `src/preditor_de_falhas_ml/features.py`; aqui ela é aplicada aos dados locais.

## Objetivo

Recalcular e conferir a variável-alvo categórica **Y** para cada registro curated, usando perda de pacotes e latência. A fonte canônica é `curated/log_rede.csv` no bucket `preditor-falhas-ml`; para executar este notebook, mantenha uma cópia local em `data/curated/log_rede.csv`.

| Prioridade | Condição | Rótulo atribuído a Y |
|---:|---|---|
| 1 | `perda_pacotes_pct > 15` | `FALHA` |
| 2 | Se a primeira condição não ocorrer e `latencia_ms > 100` | `RISCO` |
| 3 | Se nenhuma das condições anteriores ocorrer | `OK` |

Os limites são estritos: valores iguais a 15% e 100 ms não ultrapassam os limites. A regra de perda tem prioridade quando as duas métricas ultrapassam seus limites. Valores ausentes seguem o comportamento da função oficial do projeto.

O escopo é recalcular `status_real` em uma cópia em memória, comparar com o rótulo já presente no curated e resumir as classes. O CSV original não é alterado, nada é gravado no S3 e nenhum modelo é treinado.

## Entrega mínima

- Explicar as condições, os limites e a prioridade dos rótulos.
- Carregar a cópia local de `curated/log_rede.csv` e conferir as colunas usadas no cálculo.
- Recalcular `status_real` em uma cópia dos dados e comparar com o rótulo existente.
- Exibir uma amostra, a distribuição das classes e casos de demonstração.

## 1. Identificação da equipe

| Integrante | Responsabilidade registrada no projeto |
|---|---|
| Guilherme Leite Tavares | Arquitetura de dados, quadro Kanban, especificação e revisão |
| Alexandre Tiago de Oliveira | N/A |
| Ingrid Ferreira de Sousa | N/A |
| Kauan Garcia Dias de Oliveira | Revisão do notebook / aprovação do PR |
| Lucas Eduardo Malachias Bagatela | N/A |
| Stephanie Vitoria Bessa dos Santos | Revisão / aprovação de PRs

## 2. Decisões da equipe

| Decisão | Justificativa |
|---|---|
| Usar `data/curated/log_rede.csv` como entrada local | É uma cópia do objeto curated canônico `curated/log_rede.csv` do bucket `preditor-falhas-ml`. |
| Manter a regra em `status_real()` no pacote | Evita duplicar a regra de negócio no notebook e permite reutilizá-la pelo projeto. |
| Recalcular `status_real` em uma cópia em memória | Preserva o CSV curated e permite comparar o rótulo existente com a regra oficial. |
| Usar as classes `FALHA`, `RISCO` e `OK` | Segue a regra de rotulagem definida para o projeto. |
| Não treinar um modelo neste notebook | Esta entrega documenta somente o cálculo da variável-alvo Y. |

## 3. Bibliotecas e ambiente

Na raiz do repositório, execute `uv sync` e selecione no editor o kernel Python desse ambiente. O projeto já inclui `pandas` e a função de rotulagem; este notebook não adiciona dependências.

## 4. Carregar e inspecionar os dados

Salve uma cópia do objeto S3 `curated/log_rede.csv` em `data/curated/log_rede.csv`. A célula abaixo localiza o repositório, carrega o CSV canônico com pandas e apresenta uma amostra inicial.

In [ ]:
from pathlib import Path

import pandas as pd

from preditor_de_falhas_ml import status_real

raiz_repo = Path.cwd()
if not (raiz_repo / "data" / "curated" / "log_rede.csv").is_file():
    raiz_repo = raiz_repo.parent

caminho_dados = raiz_repo / "data" / "curated" / "log_rede.csv"
if not caminho_dados.is_file():
    raise FileNotFoundError(
        f"Não encontrei {caminho_dados}. "
        "Baixe curated/log_rede.csv do bucket preditor-falhas-ml e salve em data/curated/."
    )

dados = pd.read_csv(caminho_dados)
print(f"Arquivo: {caminho_dados}")
print(f"Registros: {len(dados):,}")
display(dados.head())

## 5. Conferir as métricas necessárias

O cálculo depende de `perda_pacotes_pct` e `latencia_ms`; o contrato curated também inclui `status_real`. As métricas são convertidas para números na cópia em memória, mantendo o CSV original intacto.

In [ ]:
colunas_metricas = ["perda_pacotes_pct", "latencia_ms"]
colunas_requeridas = ["timestamp", "ip", *colunas_metricas, "status_real"]
colunas_ausentes = [
    coluna for coluna in colunas_requeridas if coluna not in dados.columns
]

if colunas_ausentes:
    raise ValueError(
        "O arquivo não contém as colunas necessárias: " + ", ".join(colunas_ausentes)
    )

rotulos_originais = dados["status_real"].astype("string")
dados_rotulados = dados.copy()
for coluna in colunas_metricas:
    dados_rotulados[coluna] = pd.to_numeric(dados_rotulados[coluna], errors="raise")

print("Colunas validadas:", ", ".join(colunas_requeridas))

## 6. Calcular Y com a função do projeto

A função auxiliar converte valores ausentes do pandas para `None` e chama `status_real()`. Os limites e a prioridade ficam definidos uma única vez em `src/preditor_de_falhas_ml/features.py`. Depois do recálculo, o notebook mostra quantos rótulos curated diferem do resultado oficial.

In [ ]:
def rotular_linha(linha: pd.Series) -> str:
    perda = (
        None
        if pd.isna(linha["perda_pacotes_pct"])
        else float(linha["perda_pacotes_pct"])
    )
    latencia = None if pd.isna(linha["latencia_ms"]) else float(linha["latencia_ms"])
    return status_real(perda, latencia)


dados_rotulados["status_real"] = dados_rotulados.apply(rotular_linha, axis=1)
divergentes = rotulos_originais.ne(dados_rotulados["status_real"]).fillna(True)
print(f"Rótulos divergentes em relação ao curated: {int(divergentes.sum())}")
colunas_exibicao = ["timestamp", "ip", *colunas_metricas, "status_real"]
display(dados_rotulados[colunas_exibicao].head(10))

## 7. Conferir a distribuição dos rótulos

A tabela mostra a quantidade e o percentual de registros em cada classe, sempre na ordem `FALHA`, `RISCO` e `OK`.

In [ ]:
ordem_rotulos = ["FALHA", "RISCO", "OK"]
resumo = (
    dados_rotulados["status_real"]
    .value_counts()
    .reindex(ordem_rotulos, fill_value=0)
    .rename_axis("status_real")
    .to_frame("quantidade")
)
resumo["percentual_pct"] = (resumo["quantidade"] / len(dados_rotulados) * 100).round(2)

display(resumo)

## 8. Demonstrar limites, prioridade e valores ausentes

Os exemplos abaixo chamam a mesma função do projeto. Eles incluem os limites exatos, cada condição acima do limite, as duas condições verdadeiras ao mesmo tempo e um caso sem métricas disponíveis.

In [ ]:
casos_exemplo = pd.DataFrame(
    {
        "caso": [
            "Nos limites exatos",
            "Perda acima do limite",
            "Perda e latência acima dos limites",
            "Somente latência acima do limite",
            "Sem métricas disponíveis",
        ],
        "perda_pacotes_pct": [15.0, 15.1, 20.0, 10.0, pd.NA],
        "latencia_ms": [100.0, 100.0, 120.0, 100.1, pd.NA],
    }
)

casos_exemplo["status_real"] = casos_exemplo.apply(rotular_linha, axis=1)
display(casos_exemplo)

## 9. Checklist da equipe

- [x] Regras de Y, limites estritos e prioridade descritos.
- [x] Notebook chama a função oficial do pacote para criar `status_real`.
- [x] CSV original preservado; rótulo criado em uma cópia em memória.
- [x] Amostra dos dados e distribuição das classes incluídas.
- [x] Casos de limite, prioridade e ausência de métricas incluídos.
- [ ] Executar as células no ambiente do projeto e revisar os resultados antes da entrega.

## Registro final

Ao executar todas as células em sequência, o notebook recalcula `status_real` em memória a partir de `curated/log_rede.csv`, informa divergências com o rótulo existente e apresenta a amostra, o resumo das classes e os exemplos da regra.